Manual Preprocessing: Mask Irregular
====================================

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

from pathlib import Path
import autogalaxy as ag
import autogalaxy.plot as aplt
import numpy as np

This tool allows one to mask a bespoke mask for a given image of a galaxy, which can then be loaded
before a model-fit.

This tool creates an irregular mask, which can form any shape and is not restricted to circles, annuli, ellipses,
etc. This mask is created as follows:

1) Blur the observed image with a Gaussian kernel of specified FWHM.
2) Compute the absolute S/N map of that blurred image and the noise-map.
3) Create the mask for all pixels with a S/N above a theshold value.

For galaxies without a lens light component this masks create a source-only mask. If the galaxy light is included
it includes the galaxy light .

The following parameters determine the behaviour of this function:

The sigma value (e.g. FWHM) of the Gaussian the image is blurred with and the S/N threshold defining above which a 
image-pixel value must be to not be masked.

In [ ]:
blurring_gaussian_sigma = 0.1
snr_cut = 10.0

Setup the path the datasets we'll use to illustrate preprocessing, which is the 
folder `dataset/data_preparation/imaging/simple`.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "imaging" / dataset_name

data = ag.Array2D.from_fits(file_path=dataset_path / "data.fits", pixel_scales=0.1)
noise_map = ag.Array2D.from_fits(
    file_path=dataset_path / "noise_map.fits", pixel_scales=0.1
)

Returns the 2D Gaussian that the image is blurred with. This blurring smooths over noise in the image, which will 
otherwise lead unmasked values with in individual pixels if not smoothed over correctly.

In [ ]:
blurring_gaussian = ag.Convolver.from_gaussian(
    shape_native=(31, 31),
    pixel_scales=data.pixel_scales,
    sigma=blurring_gaussian_sigma,
)

Blur the image with this Gaussian smoothing kernel and plot the resulting image.

In [ ]:
blurred_image = blurring_gaussian.convolved_image_from(image=data, blurring_image=None)
aplt.plot_array(array=blurred_image, title="Image")

Now compute the absolute signal-to-noise map of this blurred image, given the noise-map of the observed dataset.

In [ ]:
blurred_signal_to_noise_map = blurred_image / noise_map
aplt.plot_array(array=blurred_signal_to_noise_map, title="Image")

Now create the mask in 2ll pixels where the signal to noise is above some threshold value.

In [ ]:
mask = ag.Mask2D(
    mask=np.where(blurred_signal_to_noise_map.native > snr_cut, False, True),
    pixel_scales=data.pixel_scales,
)

aplt.plot_array(array=data, title="Data")

Now we`re happy with the mask, lets output it to the dataset folder of the galaxy, so that we can load it from a .fits
file in our pipelines!

In [ ]:
aplt.fits_array(array=mask, file_path=Path(dataset_path, "mask.fits"), overwrite=True)
